In [30]:
import sys
!{sys.executable} -m pip install pillow

e:\RAG\.venv\Scripts\python.exe: No module named pip


In [31]:
import fitz  # PyMuPDF
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

In [32]:
###Clip Model
import os
from dotenv import load_dotenv
load_dotenv()

## set up the environment
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

### initialize the Clip Model for unified embeddings
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 34737.24it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [33]:
### Embedding functions (always return 512-d vectors)
def _as_512_vector(model_out) -> np.ndarray:
    """Extract a (512,) float32 vector from either a tensor or a Transformers ModelOutput."""
    if hasattr(model_out, "pooler_output") and model_out.pooler_output is not None:
        vec = model_out.pooler_output
    else:
        vec = model_out
    if isinstance(vec, np.ndarray):
        arr = vec
    else:
        # torch.Tensor or similar
        arr = vec.detach().cpu().numpy()
    arr = np.asarray(arr).astype(np.float32)
    arr = arr.squeeze()
    # If we somehow got token embeddings (seq, 512), mean-pool to (512,)
    if arr.ndim == 2:
        arr = arr.mean(axis=0)
    if arr.shape != (512,):
        raise ValueError(f"Expected a 512-d vector, got shape {arr.shape}")
    arr = arr / (np.linalg.norm(arr) + 1e-12)
    return arr


def embed_image(image_data):
    """Embed image using CLIP; returns np.ndarray shape (512,) float32."""
    if isinstance(image_data, str):
        image = Image.open(image_data).convert("RGB")
    else:
        image = image_data
    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        out = clip_model.get_image_features(**inputs)
    return _as_512_vector(out)


def embed_text(text):
    """Embed text using CLIP; returns np.ndarray shape (512,) float32."""
    inputs = clip_processor(
        text=[text],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77,
    )
    with torch.no_grad():
        out = clip_model.get_text_features(**inputs)
    return _as_512_vector(out)

In [34]:
## Process PDF
pdf_path="E:/RAG/multimodal_sample.pdf"
doc=fitz.open(pdf_path)
# Storage for all documents and embeddings
all_docs = []
all_embeddings = []
image_data_store = {}  # Store actual image data for LLM

# Text splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [35]:
doc

Document('E:/RAG/multimodal_sample.pdf')

In [36]:
for i,page in enumerate(doc):
    ## process text
    text=page.get_text()
    if text.strip():
        ##create temporary document for splitting
        temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"})
        text_chunks = splitter.split_documents([temp_doc])

        #Embed each chunk using CLIP
        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content)
            all_embeddings.append(embedding)
            all_docs.append(chunk)



    ## process images
    ##Three Important Actions:

    ##Convert PDF image to PIL format
    ##Store as base64 for GPT-4V (which needs base64 images)
    ##Create CLIP embedding for retrieval

    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            
            # Convert to PIL Image
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            
            # Create unique identifier
            image_id = f"page_{i}_img_{img_index}"
            
            # Store image as base64 for later use with GPT-4V
            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode()
            image_data_store[image_id] = img_base64
            
            # Embed image using CLIP
            embedding = embed_image(pil_image)
            all_embeddings.append(embedding)
            
            # Create document for image
            image_doc = Document(
                page_content=f"[Image: {image_id}]",
                metadata={"page": i, "type": "image", "image_id": image_id}
            )
            all_docs.append(image_doc)
            
        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue

doc.close()

In [37]:
all_docs

[Document(metadata={'page': 0, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.'),
 Document(metadata={'page': 0, 'type': 'image', 'image_id': 'page_0_img_0'}, page_content='[Image: page_0_img_0]')]

In [38]:
image_data_store

{'page_0_img_0': 'iVBORw0KGgoAAAANSUhEUgAAAZAAAAEsCAIAAABi1XKVAAAEGUlEQVR4nO3W0QkCQRAFwVu5vMXIxxQOBPdaqyJ4zEcza2YOgILH7gEAVwkWkCFYQIZgARmCBWQIFpAhWECGYAEZggVkCBaQIVhAhmABGYIFZAgWkCFYQIZgARmCBWQIFpAhWECGYAEZggVkCBaQIVhAhmABGYIFZAgWkCFYQIZgARmCBWQIFpAhWECGYAEZggVkCBaQIVhAhmABGYIFZAgWkCFYQIZgARmCBWQIFpAhWECGYAEZggVkCBaQIVhAhmABGYIFZJy7B8AXrXXTc8/sXtDgwwIyBAvIECwgQ7CADMECMgQLyBAsIEOwgAzBAjIEC8gQLCBDsIAMwQIyBAvIECwgQ7CADMECMgQLyBAsIEOwgAzBAjIEC8gQLCBDsIAMwQIyBAvIECwgQ7CADMECMgQLyBAsIEOwgAzBAjIEC8gQLCBDsIAMwQIyBAvIECwgQ7CADMECMgQLyBAsIEOwgAzBAjIEC8gQLCBDsIAMwQIyzt0D+NR6rXsecZ6zewK/xocFZAgWkCFYQIZgARmCBWQIFpAhWECGYAEZggVkCBaQIVhAhmABGYIFZAgWkCFYQIZgARmCBWQIFpAhWECGYAEZggVkCBaQIVhAhmABGYIFZAgWkCFYQIZgARmCBWQIFpAhWECGYAEZggVkCBaQIVhAhmABGYIFZAgWkCFYQIZgARmCBWQIFpAhWECGYAEZggVkCBaQIVhAhmABGYIFZJzHLa113NPM7gXwx3xYQIZgARmCBWQIFpAhWECGYAEZggVkCBaQIVhAhmABGYIFZAgWkCFYQIZgARmCBWQIFpAhWECGYAEZggVkCBaQIVhAhmABGYIFZAgWkCFYQIZgARmCBWQIFpAhWECGYAEZggVkCBaQIVhAhmABGYIFZAgWkCFYQIZgARmCBWQIFpAhWECGYAEZggVkCB

In [39]:
# Sanity checks
print("docs:", len(all_docs))
print("raw all_embeddings count:", len(all_embeddings))
if all_embeddings:
    print("raw all_embeddings[0] shape:", np.asarray(all_embeddings[0]).shape)
print("rebuilt_embeddings count:", len(rebuilt_embeddings) if 'rebuilt_embeddings' in globals() else 0)
if 'rebuilt_embeddings' in globals() and rebuilt_embeddings:
    print("rebuilt_embeddings[0] shape:", np.asarray(rebuilt_embeddings[0]).shape)

docs: 2
raw all_embeddings count: 2
raw all_embeddings[0] shape: (512,)
rebuilt_embeddings count: 2
rebuilt_embeddings[0] shape: (512,)


In [40]:
# Debugging: Inspect the structure of all_embeddings
for idx, embedding in enumerate(all_embeddings):
    print(f"Embedding {idx}: Shape = {np.shape(embedding)}")

Embedding 0: Shape = (512,)
Embedding 1: Shape = (512,)


In [41]:
# Rebuild embeddings aligned to `all_docs` (fixes mixed shapes like (77,512) vs (50,768))
import base64
import io
from langchain_core.embeddings import Embeddings

def _pil_from_base64(b64: str) -> Image.Image:
    return Image.open(io.BytesIO(base64.b64decode(b64))).convert("RGB")

rebuilt_embeddings = []
for doc in all_docs:
    doc_type = doc.metadata.get("type")
    if doc_type == "image":
        image_id = doc.metadata.get("image_id")
        pil_img = _pil_from_base64(image_data_store[image_id])
        rebuilt_embeddings.append(embed_image(pil_img))
    else:
        rebuilt_embeddings.append(embed_text(doc.page_content))

# Keep `all_embeddings` consistent for downstream cells
all_embeddings = rebuilt_embeddings

# (n, 512) float32 matrix for FAISS
embeddings_matrix = np.vstack(all_embeddings).astype(np.float32)
embeddings_matrix.shape

(2, 512)

In [42]:
# Create unified FAISS vector store with CLIP embeddings (text + images)
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings

class ClipQueryEmbeddings(Embeddings):
    def embed_documents(self, texts):
        return [embed_text(t).tolist() for t in texts]
    def embed_query(self, text):
        return embed_text(text).tolist()

clip_embeddings = ClipQueryEmbeddings()

# Pair each doc's page_content with its precomputed embedding
text_embeddings = [(doc.page_content, vec.tolist()) for doc, vec in zip(all_docs, rebuilt_embeddings)]
metadatas = [doc.metadata for doc in all_docs]

vectorstore = FAISS.from_embeddings(
    text_embeddings=text_embeddings,
    embedding=clip_embeddings,
    metadatas=metadatas,
)
vectorstore

In [43]:
# Sanity check: inspect what CLIP returns in this environment
t_inputs = clip_processor(text=["hello"], return_tensors="pt", padding=True, truncation=True, max_length=77)
i_inputs = clip_processor(images=Image.new("RGB", (224, 224), color="white"), return_tensors="pt")
with torch.no_grad():
    t_out = clip_model.get_text_features(**t_inputs)
    i_out = clip_model.get_image_features(**i_inputs)
print("get_text_features type:", type(t_out))
print("get_image_features type:", type(i_out))
for name, out in [("text", t_out), ("image", i_out)]:
    for attr in ["last_hidden_state", "pooler_output", "text_embeds", "image_embeds"]:
        if hasattr(out, attr):
            val = getattr(out, attr)
            try:
                shape = tuple(val.shape)
            except Exception:
                shape = "<no shape>"
            print(f"{name}.{attr} shape: {shape}")

get_text_features type: <class 'transformers.modeling_outputs.BaseModelOutputWithPooling'>
get_image_features type: <class 'transformers.modeling_outputs.BaseModelOutputWithPooling'>
text.last_hidden_state shape: (1, 3, 512)
text.pooler_output shape: (1, 512)
image.last_hidden_state shape: (1, 50, 768)
image.pooler_output shape: (1, 512)


In [50]:
# Use the unified FAISS store created above
# (Keep the name `vector_store` for the downstream pipeline cells.)
vector_store = vectorstore
vector_store

In [45]:
# Initialize GPT-4 Vision model
llm = init_chat_model("openai:gpt-4.1")
llm

ChatOpenAI(profile={'name': 'GPT-4.1', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000022E8C08A550>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000022E8CACBE10>, root_client=<openai.OpenAI object at 0x0000022E8C08A050>, root_async_client=<openai.AsyncOpenAI object at 0x0000022E8CACB910>, model_name='gpt-4.1', model_kwargs={}, openai_api_key=SecretStr('**

In [51]:
def retrieve_multimodal(query, k=5):
    """Unified retrieval using CLIP embeddings for both text and images."""
    # Embed query using CLIP
    query_embedding = embed_text(query).tolist()
    
    # Search in unified vector store
    results = vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k
    )
    
    return results

In [47]:
def create_multimodal_message(query, retrieved_docs):
    """Create a message with both text and images for GPT-4V."""
    content = []
    
    # Add the query
    content.append({
        "type": "text",
        "text": f"Question: {query}\n\nContext:\n"
    })
    
    # Separate text and image documents
    text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "text"]
    image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]
    
    # Add text context
    if text_docs:
        text_context = "\n\n".join([
            f"[Page {doc.metadata['page']}]: {doc.page_content}"
            for doc in text_docs
        ])
        content.append({
            "type": "text",
            "text": f"Text excerpts:\n{text_context}\n"
        })
    
    # Add images
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id and image_id in image_data_store:
            content.append({
                "type": "text",
                "text": f"\n[Image from page {doc.metadata['page']}]:\n"
            })
            content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{image_data_store[image_id]}"
                }
            })
    
    # Add instruction
    content.append({
        "type": "text",
        "text": "\n\nPlease answer the question based on the provided text and images."
    })
    
    return HumanMessage(content=content)

In [ ]:
def multimodal_pdf_rag_pipeline(query):
    """Main pipeline for multimodal RAG."""
    # Retrieve relevant documents
    context_docs = retrieve_multimodal(query, k=5)
    
    # Print retrieved context info (useful even if the LLM call fails)
    print(f"\nRetrieved {len(context_docs)} documents:")
    for doc in context_docs:
        doc_type = doc.metadata.get("type", "unknown")
        page = doc.metadata.get("page", "?")
        if doc_type == "text":
            preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
            print(f"  - Text from page {page}: {preview}")
        else:
            print(f"  - Image from page {page}")
    print("\n")
    
    # Create multimodal message
    message = create_multimodal_message(query, context_docs)
    
    # Get response from GPT-4V (guard against missing quota / auth / model access)
    try:
        response = llm.invoke([message])
        return response.content
    except Exception as e:
        return f"LLM call failed: {type(e).__name__}: {e}"

In [52]:
if __name__ == "__main__":
    # Example queries
    queries = [
        "What does the chart on page 1 show about revenue trends?",
        "Summarize the main findings from the document",
        "What visual elements are present in the document?"
    ]
    
    for query in queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        answer = multimodal_pdf_rag_pipeline(query)
        print(f"Answer: {answer}")
        print("=" * 70)


Query: What does the chart on page 1 show about revenue trends?
--------------------------------------------------


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}